# 03 — Train Emulators (Multi-z): Cluster Profiles

All cluster profile statistics: CGD, CGED, CPP, CTP, CEP, CEEP, CMP, CYP.
Multi-snapshot emulators (z ≤ 0.5, snapshots 415–624).
All models saved to `../models/<PROFILE>_multiz/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
import matplotlib.cm as cm
import os

from cosmo_hydro_emu.pca import *
from cosmo_hydro_emu.viz import *
from cosmo_hydro_emu.load_hacc import *
from cosmo_hydro_emu.emu import *
from cosmo_hydro_emu.gp import *
from cosmo_hydro_emu.snapshot_utils import SNAPSHOT_IDS, get_snapshot_redshifts

## Configuration

In [ ]:
DirIn = '../data/400MPC_RUNS_5SG_2COSMO_PARAM/HAvoCC/'

start_sim_idx = 1
num_sims = 39
exp_variance = 0.999

z_initial = 200

do_train = True

seed_mass_scale = 1e6
vkin_scale = 1e4
eps_scale = 1e1

## Load parameters

In [ ]:
fileIn = '../data/FinalDesign.txt'
params_all = np.loadtxt(fileIn, delimiter=",", skiprows=1)
params32 = params_all[start_sim_idx - 1 : start_sim_idx - 1 + num_sims]
params32[:, 2] /= seed_mass_scale
params32[:, 3] /= vkin_scale
params32[:, 4] /= eps_scale

print('params32 shape:', params32.shape)

# Train/test split
test_sim_indices = np.array([3, 11, 19, 27, 35])
train_sim_indices = np.array([i for i in range(num_sims) if i not in test_sim_indices])

params_train = params32[train_sim_indices]
params_test = params32[test_sim_indices]

print(f'Train: {len(train_sim_indices)} sims, Test: {len(test_sim_indices)} sims')

## Snapshot setup

In [ ]:
z_all, a_all = get_snapshot_redshifts(SNAPSHOT_IDS, z_initial=z_initial)

print(f'Number of snapshots: {len(SNAPSHOT_IDS)}')
print(f'Snapshot IDs: {SNAPSHOT_IDS}')
print(f'Redshift range: z = {z_all[-1]:.2f} to {z_all[0]:.2f}')
print(f'Scale factor range: a = {a_all[0]:.3f} to {a_all[-1]:.3f}')

## Load all profile data

In [ ]:
from cosmo_hydro_emu.load_hacc import read_profile_all_snaps

PROFILE_CONFIGS = {
    'CGD':  {'file_prefix': 'ClusterGasDensityProfile',         'label': 'Cluster Gas Density'},
    'CGED': {'file_prefix': 'ClusterGasElectronDensityProfile',  'label': 'Cluster Gas Electron Density'},
    'CPP':  {'file_prefix': 'ClusterGasPressureProfile',         'label': 'Cluster Gas Pressure'},
    'CTP':  {'file_prefix': 'ClusterGasTemperatureProfile',      'label': 'Cluster Gas Temperature'},
    'CEP':  {'file_prefix': 'ClusterGasEntropyProfile',          'label': 'Cluster Gas Entropy'},
    'CEEP': {'file_prefix': 'ClusterElectronEntropyProfile',     'label': 'Cluster Electron Entropy'},
    'CMP':  {'file_prefix': 'ClusterGasMetallicityProfile',      'label': 'Cluster Gas Metallicity'},
    'CYP':  {'file_prefix': 'ClusterGasComptonYProfile',         'label': 'Cluster Compton-y (tSZ)'},
}

profile_data = {}
for short_name, config in PROFILE_CONFIGS.items():
    radius, arr = read_profile_all_snaps(DirIn, num_sims, SNAPSHOT_IDS,
                                          config['file_prefix'],
                                          start_sim_idx=start_sim_idx)
    profile_data[short_name] = arr
    print(f"{short_name}: {arr.shape}")

print(f"Radius bins: {radius.shape}")

## Radius cut

In [ ]:
rlim1, rlim2 = mass_conds('CGD')  # same limits for all profiles
rad_cond = np.where((radius > rlim1) & (radius < rlim2))[0]
y_ind_profiles = radius[rad_cond]
print(f"Radius cut: {len(rad_cond)} bins in [{rlim1:.3f}, {rlim2:.3f}]")

## Train all profiles

In [ ]:
profile_z_start_idx = 6  # z <= 0.5
z_index_range = np.arange(profile_z_start_idx, len(SNAPSHOT_IDS))
profile_z_all = z_all[profile_z_start_idx:]

profile_models = {}

for short_name, config in PROFILE_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Training {short_name}: {config['label']}")
    print(f"{'='*60}")

    arr = profile_data[short_name]
    y_vals = arr[:, :, rad_cond]  # (num_sims, num_snaps, n_bins_cut)

    model_dir = f'../models/{short_name}_multiz/'

    if do_train:
        os.makedirs(model_dir, exist_ok=True)
        do_gp_train_multiple(
            model_dir=model_dir,
            p_train_all=params_train,
            y_vals_all=y_vals[train_sim_indices],
            y_ind_all=y_ind_profiles,
            z_index_range=z_index_range,
            exp_variance=exp_variance
        )

    model_list, data_list = load_model_multiple(
        model_dir=model_dir,
        p_train_all=params_train,
        y_vals_all=y_vals[train_sim_indices],
        y_ind_all=y_ind_profiles,
        z_index_range=z_index_range,
    )

    profile_models[short_name] = (model_list, data_list)
    print(f"  Loaded {len(model_list)} models for {short_name}")

## Validation at z=0 (all profiles)

In [ ]:
n_profiles = len(PROFILE_CONFIGS)
ncols = 4
nrows = (n_profiles + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))
axes = axes.flatten()

for idx, (short_name, config) in enumerate(PROFILE_CONFIGS.items()):
    ax = axes[idx]
    model_list, data_list = profile_models[short_name]
    arr = profile_data[short_name]
    y_vals = arr[:, :, rad_cond]

    for t_idx in test_sim_indices[:3]:
        target = y_vals[t_idx, -1, :]  # z=0 (last snapshot)
        pred_mean, pred_quant = emulate(model_list[-1], params32[t_idx])
        ax.plot(y_ind_profiles, target, 'k-', alpha=0.5)
        ax.plot(y_ind_profiles, pred_mean[:, 0], 'r--', alpha=0.7)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(short_name)
    ax.set_xlabel(r'$r/R_{500c}$')

for idx in range(len(PROFILE_CONFIGS), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig('../plots/profiles_multiz_validation.png', bbox_inches='tight')

## Redshift interpolation test

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

test_params = params32[test_sim_indices[0]]
z_grid = np.linspace(profile_z_all[-2], profile_z_all[1], 5)

for idx, (short_name, config) in enumerate(PROFILE_CONFIGS.items()):
    ax = axes[idx]
    model_list, data_list = profile_models[short_name]

    for z_target in z_grid:
        params_with_z = np.append(test_params, [z_target])[np.newaxis, :]
        pred_z, pred_z_err = emu_redshift(params_with_z, model_list, data_list, profile_z_all)
        ax.plot(y_ind_profiles, pred_z[:, 0], label=f'z={z_target:.2f}')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(short_name)
    ax.legend(fontsize=6)

plt.tight_layout()
plt.savefig('../plots/profiles_multiz_redshift_interp.png', bbox_inches='tight')